# Lab 19: API Requests and Caching — Analysis

Run these experiments to see the difference caching makes.

**Before you start:** Make sure your `crypto.py` implementations pass all tests.

**Setup:** Put your CoinGecko Demo API key in the cell below.

In [1]:
import sys
sys.path.insert(0, '../src')

import time
from crypto import get_price, get_prices_batch, CoinCache, get_price_cached

API_KEY = "CG-aJ7i8wYuMcHrV4XfRZKdWnYf"  # Replace with your CoinGecko Demo key

## Experiment 1: Uncached vs. Cached

Fetch Bitcoin's price 10 times — first without caching, then with caching.
Compare the total time and number of API calls.

In [2]:
# --- Uncached: 10 direct API calls ---
start = time.time()
for i in range(10):
    price = get_price("bitcoin", API_KEY)
uncached_time = time.time() - start

print(f"Uncached: 10 requests in {uncached_time:.2f} seconds")
print(f"Last price: ${price:,.2f}")

Uncached: 10 requests in 0.75 seconds
Last price: $71,776.00


In [4]:
# --- Cached: 10 lookups through cache ---
cache = CoinCache(ttl_seconds=60)

start = time.time()
for i in range(10):
    price = get_price_cached("bitcoin", API_KEY, cache)
cached_time = time.time() - start

print(f"Cached: 10 lookups in {cached_time:.2f} seconds")
print(f"Cache hits: {cache.hits}, Cache misses: {cache.misses}")
print(f"Speedup: {uncached_time / cached_time:.1f}x faster")

Cached: 10 lookups in 0.05 seconds
Cache hits: 9, Cache misses: 1
Speedup: 14.5x faster


### Writeup Questions — Experiment 1

1. How many API calls did the cached version actually make? Why that number?
2. What was the approximate speedup? Why is the difference so large?
3. Is there any downside to this speedup? What are you giving up?

*Your answers here:*

1. just 1, because the data was already saved in a cache.
2. The diffference was because the data was already saved locally, much qiucker to access.
3. you aree giving up live, realtime data. 

## Experiment 2: TTL Exploration

Try three different TTL values. For each one, simulate a pattern
of lookups spaced 2 seconds apart and observe the hit rate.

In [13]:
ttl_values = [1, 5, 30]

for ttl in ttl_values:
    cache = CoinCache(ttl_seconds=ttl)
    
    for i in range(6):
        price = get_price_cached("bitcoin", API_KEY, cache)
        if i < 5:  # Don't sleep after last lookup
            time.sleep(2)
    
    total = cache.hits + cache.misses
    hit_rate = cache.hits / total * 100
    print(f"TTL={ttl:2d}s: {cache.hits} hits, {cache.misses} misses, hit rate={hit_rate:.0f}%")

TTL= 1s: 0 hits, 6 misses, hit rate=0%
TTL= 5s: 4 hits, 2 misses, hit rate=67%
TTL=30s: 5 hits, 1 misses, hit rate=83%


### Writeup Questions — Experiment 2

1. With TTL=1 second and lookups every 2 seconds, what hit rate do you expect? Did the results match?
2. If you were building a portfolio tracker that updates every time you open the app, what TTL would you choose? Explain your reasoning.
3. Is there a scenario where you'd want a TTL of 0 (no caching at all)? What about a TTL of infinity (cache forever)?

*Your answers here:*

1. You should be getting a 0% hit rate by the time the lookup happens the cache has already expired, the results showed 0%
2.  You would want a good balance between performance and real time data so somthing around 10 seconds would be a good ttl. 
3. if the data ever needs to be perfect up to date then you wouldf use a ttl of 0. A cache of forever would only be good for data that never change or rerely changes.

## Experiment 3: Batch Efficiency

Compare fetching 5 coins one at a time vs. in a single batch request.

In [5]:
coins = ["bitcoin", "ethereum", "solana", "cardano", "dogecoin"]

# --- Individual calls ---
start = time.time()
individual_prices = {}
for coin in coins:
    individual_prices[coin] = get_price(coin, API_KEY)
individual_time = time.time() - start

print(f"Individual: {len(coins)} calls in {individual_time:.2f} seconds")

# --- Batch call ---
start = time.time()
batch_prices = get_prices_batch(coins, API_KEY)
batch_time = time.time() - start

print(f"Batch: 1 call in {batch_time:.2f} seconds")
print(f"Speedup: {individual_time / batch_time:.1f}x faster")

# Show the prices
print("\nPrices:")
for coin, price in batch_prices.items():
    print(f"  {coin}: ${price:,.2f}")

Individual: 5 calls in 0.61 seconds
Batch: 1 call in 0.11 seconds
Speedup: 5.8x faster

Prices:
  bitcoin: $71,787.00
  ethereum: $2,187.46
  solana: $83.20
  cardano: $0.25
  dogecoin: $0.09


### Writeup Questions — Experiment 3

1. How much faster was the batch call? Where does that time saving come from?
2. Batching and caching are both ways to reduce API calls. When would you use one vs. the other? Can you use both?

*Your answers here:*

1. A bach is faster by about half a second. you only wait once for the call so you save time on less latency. 
2. you would use batching when you have to get multiple peices of data at the same time. You would use caching when the same data that you might request repeatedly over time. 